In [ ]:
# Este notebook registra o conector para escutar a tabela do PostgreSQL.

import requests
import json

# URL do Kafka Connect (Debezium) exposto no docker-compose
DEBEZIUM_URL = "http://debezium:8083/connectors"

# Configuração do conector focado no Cenário E-commerce
config_debezium = {
    "name": "postgres-cdc-ecommerce",
    "config": {
        "connector.class": "io.debezium.connector.postgresql.PostgresConnector",
        "tasks.max": "1",
        "plugin.name": "pgoutput", # Plugin nativo do Postgres 15 para replicação
        
        # Conexão com o Banco de Dados interno do Docker
        "database.hostname": "postgres",
        "database.port": "5432",
        "database.user": "postgres",
        "database.password": "senha123",
        "database.dbname": "postgres",
        
        # Identificador do Cluster para o Kafka criar os tópicos
        "topic.prefix": "cdc", 
        
        # Filtro: Monitorar apenas a tabela transacoes_financeiras
        "table.include.list": "public.transacoes_financeiras",
        
        "publication.name": "dbz_publication",
        "publication.autocreate.mode": "disabled",

        "key.converter.schemas.enable": "false",
        "value.converter.schemas.enable": "false"

    }
}

In [13]:
# Enviando o comando POST para registrar o conector

print("Enviando configuração para o Debezium...")
response = requests.post(
    DEBEZIUM_URL, 
    headers={"Content-Type": "application/json"}, 
    data=json.dumps(config_debezium)
)

if response.status_code == 201:
    print("Conector de CDC criado com sucesso!")
else:
    print(f"Erro ao criar conector: {response.status_code}")
    print(response.text)

# %%
# Validação: Listar conectores ativos
ativos = requests.get(DEBEZIUM_URL).json()
print(f"Conectores ativos no momento: {ativos}")

Enviando configuração para o Debezium...
Conector de CDC criado com sucesso!
Conectores ativos no momento: ['postgres-cdc-ecommerce']


In [ ]:
# Resetar o conector de escutar a tabela do PostgreSQL.

import requests

DEBEZIUM_URL = "http://debezium:8083/connectors/postgres-cdc-ecommerce"

# Envia um comando DELETE para apagar o conector antigo se ele existir
print("Limpando conectores antigos...")
response = requests.delete(DEBEZIUM_URL)

if response.status_code == 204:
    print("Conector antigo removido com sucesso!")
elif response.status_code == 404:
    print("Idempotência: Nenhum conector antigo encontrado. Pronto para criar.")
else:
    print(f"Aviso: {response.status_code}")

: 